# Body Type Classifier — SOMAset Pre-training + Fine-tuning

**Two-phase training:**  
1. **Pre-train** on SOMAset (100K synthetic images, 50 identities) — learns body morphology features  
2. **Fine-tune** on body_dataset (300 real images, 3 classes: ectomorfo, endomorfo, mesomorfo)

**Output:** `body_type_model.onnx` (~8 MB) — ready for Flask inference.


## Setup

In [ ]:
!pip install torch torchvision onnx onnxruntime pillow numpy scikit-learn tqdm


In [ ]:
import os, torch, torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from PIL import Image
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

CLASS_NAMES = ['ectomorfo', 'endomorfo', 'mesomorfo']
NUM_CLASSES = len(CLASS_NAMES)
SOMASET_CLASSES = 50  # 50 identities in SOMAset
SOMASET_DIR = '/content/somaset'
DATA_DIR = '/content/body_dataset'



## Upload Datasets

Upload both `somaset.zip` and `body_dataset.zip` to Colab.

> **SOMAset** is ~330 MB (100K images). For Drive: `!cp -r /content/drive/MyDrive/somaset /content/somaset`

In [ ]:
from google.colab import files
import zipfile, io

print('Upload body_dataset.zip (~5 MB):')
uploaded = files.upload()
for fn in uploaded:
    with zipfile.ZipFile(io.BytesIO(uploaded[fn]), 'r') as z:
        z.extractall('/content')
    print('Extracted:', fn)

if not os.path.exists(SOMASET_DIR):
    print('Upload somaset.zip (~330 MB) — press Cancel if already on Drive:')
    uploaded = files.upload()
    for fn in uploaded:
        with zipfile.ZipFile(io.BytesIO(uploaded[fn]), 'r') as z:
            z.extractall('/content')
        print('Extracted:', fn)
else:
    print('SOMAset already present at', SOMASET_DIR)

# Verify body_dataset
for cls in CLASS_NAMES:
    path = os.path.join(DATA_DIR, cls)
    imgs = [f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print('  %s: %d images' % (cls, len(imgs)))


## Phase 1: SOMAset Pre-training

Loads SOMAset images from subject folders `01/`…`50/`. Each subject is a class label (0–49).
This teaches the model structural body features (height, proportions, build) that transfer to body-type classification.

In [ ]:
class SOMAsetDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        img = self.transform(img)
        return img, self.labels[idx]


# Scan SOMAset — collect all image paths with subject label (0-49)
soma_paths, soma_labels = [], []
for subj_dir in sorted(os.listdir(SOMASET_DIR)):
    subj_path = os.path.join(SOMASET_DIR, subj_dir)
    if not os.path.isdir(subj_path):
        continue
    subj_id = int(subj_dir) - 1  # 01 -> 0, 50 -> 49
    for root, _, files in os.walk(subj_path):
        for fn in sorted(files):
            if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                soma_paths.append(os.path.join(root, fn))
                soma_labels.append(subj_id)

soma_paths = np.array(soma_paths)
soma_labels = np.array(soma_labels)
print('SOMAset images: %d' % len(soma_paths))
print('Subjects: %d' % len(np.unique(soma_labels)))

# 80/20 stratified split (no test needed for pre-training)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(soma_paths, soma_labels))

soma_train_paths = soma_paths[train_idx]
soma_train_labels = soma_labels[train_idx]
soma_val_paths = soma_paths[val_idx]
soma_val_labels = soma_labels[val_idx]
print('Train: %d | Val: %d' % (len(soma_train_paths), len(soma_val_paths)))

# Transforms
soma_train_tfm = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
soma_val_tfm = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

ds_soma_train = SOMAsetDataset(soma_train_paths, soma_train_labels, soma_train_tfm)
ds_soma_val = SOMAsetDataset(soma_val_paths, soma_val_labels, soma_val_tfm)

dl_soma_train = DataLoader(ds_soma_train, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
dl_soma_val = DataLoader(ds_soma_val, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print('SOMAset DataLoaders ready')


In [ ]:
# Build MobileNetV3 with 50-class head for SOMAset
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)

# Freeze early layers, train last 4 blocks + classifier
for param in model.features.parameters():
    param.requires_grad = False
for i in range(-4, 0):
    for param in model.features[i].parameters():
        param.requires_grad = True

in_features = model.classifier[3].in_features
model.classifier[3] = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, SOMASET_CLASSES),
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([
    {'params': model.features[-4:].parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 1e-3},
], lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable params: %dK' % (total / 1000))


In [ ]:
SOMA_EPOCHS = 5
best_soma_acc = 0.0

for epoch in range(SOMA_EPOCHS):
    model.train()
    train_loss = 0.0; correct = 0; total = 0
    for images, labels in tqdm(dl_soma_train, desc='SOMA Epoch %d/%d' % (epoch+1, SOMA_EPOCHS)):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    scheduler.step()

    model.eval()
    val_correct = 0; val_total = 0
    with torch.no_grad():
        for images, labels in dl_soma_val:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, pred = outputs.max(1)
            val_total += labels.size(0)
            val_correct += pred.eq(labels).sum().item()

    val_acc = 100.0 * val_correct / val_total
    train_acc = 100.0 * correct / total
    loss_avg = train_loss / len(dl_soma_train)
    print('Loss: %.4f | Train: %.2f%% | Val: %.2f%%' % (loss_avg, train_acc, val_acc))

    if val_acc > best_soma_acc:
        best_soma_acc = val_acc
        torch.save(model.state_dict(), '/content/somaset_pretrained.pth')

print('Best SOMAset val acc: %.2f%%' % best_soma_acc)


## Phase 2: Fine-tune on body_dataset

Load the SOMAset pre-trained weights, replace classifier head with 3 classes, and fine-tune on the 300 real images.

In [ ]:
class BodyDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        img = self.transform(img)
        return img, self.labels[idx]

all_paths, all_labels = [], []
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(DATA_DIR, cls_name)
    for fn in sorted(os.listdir(cls_dir)):
        if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_paths.append(os.path.join(cls_dir, fn))
            all_labels.append(cls_idx)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)
print('Total: %d | Distribution: %s' % (len(all_paths), np.bincount(all_labels)))

# 60/20/20 stratified split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.4, random_state=42)
train_idx, temp_idx = next(sss.split(all_paths, all_labels))
paths_train, labels_train = all_paths[train_idx], all_labels[train_idx]
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(sss2.split(all_paths[temp_idx], all_labels[temp_idx]))
paths_val, labels_val = all_paths[temp_idx][val_idx], all_labels[temp_idx][val_idx]
paths_test, labels_test = all_paths[temp_idx][test_idx], all_labels[temp_idx][test_idx]
print('Train: %d | Val: %d | Test: %d' % (len(paths_train), len(paths_val), len(paths_test)))

# Strong augmentation for small dataset
train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])
val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

ds_train = BodyDataset(paths_train, labels_train, train_transform)
ds_val   = BodyDataset(paths_val,   labels_val,   val_transform)
ds_test  = BodyDataset(paths_test,  labels_test,  val_transform)

dl_train = DataLoader(ds_train, batch_size=16, shuffle=True,  num_workers=2)
dl_val   = DataLoader(ds_val,   batch_size=16, shuffle=False, num_workers=2)
dl_test  = DataLoader(ds_test,  batch_size=16, shuffle=False, num_workers=2)
print('body_dataset DataLoaders ready')


In [ ]:
# Build fresh model, load SOMAset pretrained weights, replace head
model = mobilenet_v3_small(weights=None)
in_features = mobilenet_v3_small().classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, SOMASET_CLASSES)
model.load_state_dict(torch.load('/content/somaset_pretrained.pth'))
print('Loaded SOMAset pretrained weights')

# Replace classifier for 3-class body type
model.classifier[3] = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(in_features, NUM_CLASSES),
)

# Unfreeze feature extractor with low LR
for param in model.features.parameters():
    param.requires_grad = True

model = model.to(DEVICE)

# Balanced class weights
cw = compute_class_weight('balanced', classes=np.unique(labels_train), y=labels_train)
class_weights = torch.FloatTensor(cw).to(DEVICE)
print('Class weights:', cw)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Fine-tune params: %dK' % (total / 1000))
print('Starting fine-tune with SOMAset-initialized weights...')


In [ ]:
EPOCHS = 30
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0; correct = 0; total = 0
    for images, labels in tqdm(dl_train, desc='Epoch %d/%d' % (epoch+1, EPOCHS)):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    scheduler.step()
    train_acc = 100.0 * correct / total

    model.eval()
    val_correct = 0; val_total = 0; val_loss = 0.0
    with torch.no_grad():
        for images, labels in dl_val:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100.0 * val_correct / val_total
    print('Loss: %.4f/%.4f | Train Acc: %.2f%% | Val Acc: %.2f%%' % (
        train_loss/len(dl_train), val_loss/len(dl_val), train_acc, val_acc))

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/content/best_body_type.pth')
        print('  --> Saved (best val acc: %.2f%%)' % val_acc)

print('\nBest validation accuracy: %.2f%%' % best_acc)

# Test
model.load_state_dict(torch.load('/content/best_body_type.pth'))
model.eval()
test_correct = 0; test_total = 0
all_preds, all_true = [], []
with torch.no_grad():
    for images, labels in dl_test:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

print('\nTest Accuracy: %.2f%%' % (100.0 * test_correct / test_total))
print('\nClassification Report:')
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=3))


## Export to ONNX

In [ ]:
import onnx

model.load_state_dict(torch.load('/content/best_body_type.pth'))
model.eval()

dummy = torch.randn(1, 3, 224, 224).to(DEVICE)

# Export to temp file first
torch.onnx.export(model, dummy, '/content/_body_temp.onnx',
    input_names=['input'], output_names=['output'],
    opset_version=17,
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})

# Reload and save as self-contained (no external .data file)
onnx_model = onnx.load('/content/_body_temp.onnx')
onnx.save_model(onnx_model, '/content/body_type_model.onnx', save_as_external_data=False)
os.remove('/content/_body_temp.onnx')

size = os.path.getsize('/content/body_type_model.onnx') / 1024 / 1024
print('Exported: /content/body_type_model.onnx (%.1f MB)' % size)


## Test Inference (ONNX)

In [ ]:
import onnxruntime as ort
session = ort.InferenceSession('/content/body_type_model.onnx')
input_name = session.get_inputs()[0].name

def predict(img_path):
    img = Image.open(img_path).convert('RGB')
    img = val_transform(img).unsqueeze(0).numpy()
    outputs = session.run(None, {input_name: img})[0]
    probs = nn.functional.softmax(torch.from_numpy(outputs[0]), dim=0).numpy()
    idx = np.argmax(probs)
    return CLASS_NAMES[idx], float(probs[idx])

for cls in CLASS_NAMES:
    sample = [f for f in os.listdir(os.path.join(DATA_DIR, cls))
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))][0]
    pred, conf = predict(os.path.join(DATA_DIR, cls, sample))
    print('Expected: %12s | Predicted: %12s | Confidence: %.3f' % (cls, pred, conf))


## Download the Model

Copy `body_type_model.onnx` to `server/` in your project.

In [ ]:
from google.colab import files
files.download('/content/body_type_model.onnx')
